In [71]:
import pandas as pd
import numpy as np

df = pd.read_csv('Crash_Reporting_-_Drivers_Data.csv')

/var/folders/n_/fvk81crd6fddbjh6csc8kmt80000gn/T/ipykernel_46631/3588527838.py:4: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Crash_Reporting_-_Drivers_Data.csv')


In [72]:
df.head

<bound method NDFrame.head of        Report Number Local Case Number                Agency Name  \
0         DM8479000T         210020119  Takoma Park Police Depart   
1        MCP2970000R          15045937                 MONTGOMERY   
2        MCP20160036         180040948   Montgomery County Police   
3         EJ7879003C         230048975  Gaithersburg Police Depar   
4        MCP2967004Y         230070277   Montgomery County Police   
...              ...               ...                        ...   
191479   MCP913000KG         250002502                 MONTGOMERY   
191480    DD56330099         250002385                  ROCKVILLE   
191481    EJ7888007C         250002405               GAITHERSBURG   
191482   MCP2528001Z         250000593                 MONTGOMERY   
191483   MCP29390075         250002221                 MONTGOMERY   

             ACRS Report Type         Crash Date/Time              Route Type  \
0       Property Damage Crash  05/27/2021 07:40:00 PM       

In [73]:
df.info()  # Columns, data types, and missing values
df.describe(include='all')  # Summary stats for numeric/categorical columns
df.head()  # Visualize first
df.shape


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 191484 entries, 0 to 191483
Data columns (total 39 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Report Number                  191484 non-null  object 
 1   Local Case Number              191484 non-null  object 
 2   Agency Name                    191484 non-null  object 
 3   ACRS Report Type               191484 non-null  object 
 4   Crash Date/Time                191484 non-null  object 
 5   Route Type                     172769 non-null  object 
 6   Road Name                      171347 non-null  object 
 7   Cross-Street Name              163383 non-null  object 
 8   Off-Road Description           17677 non-null   object 
 9   Municipality                   19126 non-null   object 
 10  Related Non-Motorist           6170 non-null    object 
 11  Collision Type                 190899 non-null  object 
 12  Weather                       

(191484, 39)

In [74]:
df.isnull().sum()
df.shape


(191484, 39)

In [75]:
df.drop(columns=[
    "Off-Road Description",          # 173k missing (96% of 190k rows)
    "Related Non-Motorist",          # 185k missing (irrelevant to driver-focused analysis)
    "Non-Motorist Substance Abuse",  # 186k missing (not your focus)
    "Circumstance",                  # 155k missing (vague/noisy for your questions)
    "Location",                      # Redundant (use Latitude/Longitude)
    "Cross-Street Name",             # 28k missing (use Road Name instead)
    "Municipality",                   # 172k missing (use Latitude/Longitude for geospatial analysis)
    "Road Name"
], inplace=True, errors='ignore')



In [76]:
df.isnull().sum()
df.shape


(191484, 31)

In [77]:
#replaced all unknown cells with NaN

df.replace("UNKNOWN", np.nan, inplace=True)
df.isnull().sum()
df.shape


(191484, 31)

In [78]:
# For Highly Critical Rows of the dataset

# Drop rows with missing Injury Severity (1,191 rows)
df = df.dropna(subset=["Injury Severity"])

# Impute "UNKNOWN" in other high-priority columns
high_priority_cols = ["Weather", "Traffic Control", "Surface Condition","Driver Substance Abuse", "Route Type", "Driver Distracted By"]
df[high_priority_cols] = df[high_priority_cols].fillna("UNKNOWN")

df.isnull().sum()
df.shape


/var/folders/n_/fvk81crd6fddbjh6csc8kmt80000gn/T/ipykernel_46631/3561577084.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[high_priority_cols] = df[high_priority_cols].fillna("UNKNOWN")


(190293, 31)

In [79]:
# Drop rows with below 5% missing values
df = df.dropna(subset=["Collision Type", "Light", "Vehicle Damage Extent", "Vehicle First Impact Location", "Vehicle Body Type", "Vehicle Movement", "Vehicle Going Dir"])
df.isnull().sum()



Report Number                       0
Local Case Number                   0
Agency Name                         0
ACRS Report Type                    0
Crash Date/Time                     0
Route Type                          0
Collision Type                      0
Weather                             0
Surface Condition                   0
Light                               0
Traffic Control                     0
Driver Substance Abuse              0
Person ID                           0
Driver At Fault                     0
Injury Severity                     0
Driver Distracted By                0
Drivers License State            3749
Vehicle ID                          0
Vehicle Damage Extent               0
Vehicle First Impact Location       0
Vehicle Body Type                   0
Vehicle Movement                    0
Vehicle Going Dir                   0
Speed Limit                         0
Driverless Vehicle                  0
Parked Vehicle                   1139
Vehicle Year

In [80]:
df.shape

(171452, 31)

In [81]:
# Drop rows with below 5% missing values
df = df.dropna(subset=["Parked Vehicle", "Drivers License State", "Vehicle Make", "Vehicle Model"])
df.isnull().sum()


Report Number                    0
Local Case Number                0
Agency Name                      0
ACRS Report Type                 0
Crash Date/Time                  0
Route Type                       0
Collision Type                   0
Weather                          0
Surface Condition                0
Light                            0
Traffic Control                  0
Driver Substance Abuse           0
Person ID                        0
Driver At Fault                  0
Injury Severity                  0
Driver Distracted By             0
Drivers License State            0
Vehicle ID                       0
Vehicle Damage Extent            0
Vehicle First Impact Location    0
Vehicle Body Type                0
Vehicle Movement                 0
Vehicle Going Dir                0
Speed Limit                      0
Driverless Vehicle               0
Parked Vehicle                   0
Vehicle Year                     0
Vehicle Make                     0
Vehicle Model       

In [82]:
df.shape

(166198, 31)

In [83]:
df.to_csv('cleaned_data.csv', index=False)

In [85]:
# Standardize the Weather column to uppercase
df['Weather'] = df['Weather'].str.upper().str.strip()
df['Weather'].value_counts()

Weather
CLEAR                                114286
RAINING                               18882
CLOUDY                                17035
UNKNOWN                               11337
SNOW                                   1464
RAIN                                   1309
FOGGY                                   622
WINTRY MIX                              355
OTHER                                   331
SLEET                                   202
BLOWING SNOW                            153
SEVERE WINDS                            143
FREEZING RAIN OR FREEZING DRIZZLE        27
FOG, SMOG, SMOKE                         21
BLOWING SAND, SOIL, DIRT                 15
SLEET OR HAIL                             9
SEVERE CROSSWINDS                         7
Name: count, dtype: int64

In [87]:
df['Traffic Control'].value_counts()

Traffic Control
NO CONTROLS                                                                 59659
TRAFFIC SIGNAL                                                              55089
UNKNOWN                                                                     22192
STOP SIGN                                                                   11778
No Controls                                                                  6381
Traffic Control Signal                                                       3846
FLASHING TRAFFIC SIGNAL                                                      2016
OTHER                                                                        1764
YIELD SIGN                                                                   1624
Stop Sign                                                                     657
Flashing Traffic Control Signal                                               251
PERSON                                                                        245


In [88]:
# Standardize to uppercase and remove extra spaces
df['Traffic Control'] = df['Traffic Control'].str.upper().str.strip()

# Replace similar values with a consistent category
df['Traffic Control'] = df['Traffic Control'].replace({
    'NO CONTROLS': 'NO CONTROLS',
    'No Controls': 'NO CONTROLS',
    'TRAFFIC CONTROL SIGNAL': 'TRAFFIC SIGNAL',
    'Stop Sign': 'STOP SIGN',
    'FLASHING TRAFFIC SIGNAL': 'FLASHING TRAFFIC SIGNAL',
    'Flashing Traffic Control Signal': 'FLASHING TRAFFIC SIGNAL',
    'YIELD SIGN': 'YIELD SIGN',
    'Yield Sign': 'YIELD SIGN',
    'PERSON': 'PERSON',
    'OTHER': 'OTHER',
    'OTHER SIGNAL': 'OTHER',
    'OTHER WARNING SIGN': 'OTHER',
    'LANE USE CONTROL SIGNAL': 'OTHER',
    'PEDESTRIAN CROSSING SIGN': 'OTHER',
    'RAILWAY CROSSING DEVICE': 'OTHER',
    'SCHOOL ZONE SIGN DEVICE': 'OTHER',
    'SCHOOL ZONE': 'OTHER',
    'SCHOOL ZONE SIGN': 'OTHER',
    # Add more mappings as necessary
})

df['Traffic Control'].value_counts()

Traffic Control
NO CONTROLS                                                                 66040
TRAFFIC SIGNAL                                                              58935
UNKNOWN                                                                     22192
STOP SIGN                                                                   12435
OTHER                                                                        2094
FLASHING TRAFFIC SIGNAL                                                      2016
YIELD SIGN                                                                   1728
FLASHING TRAFFIC CONTROL SIGNAL                                               251
PERSON                                                                        245
WARNING SIGN                                                                  142
OTHER PAVEMENT MARKING (EXCLUDING EDGELINES, CENTERLINES, OR LANE LINES)       43
PEDESTRIAN CROSSING                                                            37


In [90]:
# List of columns to convert to uppercase
columns_to_uppercase = [
    'Driver Substance Abuse', 'Light', 'Surface Condition', 
    'Injury Severity', 'Driver Distracted By', 
    'Vehicle Damage Extent', 'Vehicle First Impact Location', 
    'Vehicle Body Type', 'Vehicle Movement'
]

# Apply the transformation
df[columns_to_uppercase] = df[columns_to_uppercase].apply(lambda col: col.str.upper().str.strip())

# Check the first few rows to confirm
df[columns_to_uppercase].head()


,Driver Substance Abuse,Light,Surface Condition,Injury Severity,Driver Distracted By,Vehicle Damage Extent,Vehicle First Impact Location,Vehicle Body Type,Vehicle Movement
2,UNKNOWN,DAYLIGHT,UNKNOWN,NO APPARENT INJURY,NOT DISTRACTED,NO DAMAGE,SIX OCLOCK,PASSENGER CAR,BACKING
4,UNKNOWN,DARK LIGHTS ON,DRY,NO APPARENT INJURY,NOT DISTRACTED,DISABLING,TWELVE OCLOCK,PASSENGER CAR,MOVING CONSTANT SPEED
5,NONE DETECTED,DAYLIGHT,DRY,NO APPARENT INJURY,NOT DISTRACTED,FUNCTIONAL,SIX OCLOCK,(SPORT) UTILITY VEHICLE,SLOWING OR STOPPING
6,NONE DETECTED,DAYLIGHT,DRY,NO APPARENT INJURY,NOT DISTRACTED,FUNCTIONAL,NINE OCLOCK,PASSENGER CAR,MOVING CONSTANT SPEED
8,NONE DETECTED,DARK LIGHTS ON,DRY,NO APPARENT INJURY,NOT DISTRACTED,FUNCTIONAL,SIX OCLOCK,PASSENGER CAR,STOPPED IN TRAFFIC LANE


In [92]:
df['Driver Substance Abuse'].value_counts()

Driver Substance Abuse
NONE DETECTED                                          117924
UNKNOWN                                                 29950
NOT SUSPECT OF ALCOHOL USE, NOT SUSPECT OF DRUG USE     11807
ALCOHOL PRESENT                                          3777
ALCOHOL CONTRIBUTED                                      1324
SUSPECT OF ALCOHOL USE, NOT SUSPECT OF DRUG USE           352
UNKNOWN, UNKNOWN                                          265
ILLEGAL DRUG PRESENT                                      242
MEDICATION PRESENT                                        108
ILLEGAL DRUG CONTRIBUTED                                   90
COMBINED SUBSTANCE PRESENT                                 78
MEDICATION CONTRIBUTED                                     61
OTHER                                                      54
COMBINATION CONTRIBUTED                                    43
SUSPECT OF ALCOHOL USE, UNKNOWN                            33
UNKNOWN, NOT SUSPECT OF DRUG USE               

In [93]:
# Replace specific values with broader categories
df['Driver Substance Abuse'] = df['Driver Substance Abuse'].replace({
    'NONE DETECTED': 'NONE/NOT SUSPECTED',
    'NOT SUSPECT OF ALCOHOL USE, NOT SUSPECT OF DRUG USE': 'NONE/NOT SUSPECTED',
    'NOT SUSPECT OF ALCOHOL USE, UNKNOWN': 'NONE/NOT SUSPECTED',
    'ALCOHOL PRESENT': 'ALCOHOL',
    'ALCOHOL CONTRIBUTED': 'ALCOHOL',
    'SUSPECT OF ALCOHOL USE, NOT SUSPECT OF DRUG USE': 'ALCOHOL',
    'ILLEGAL DRUG PRESENT': 'DRUGS',
    'ILLEGAL DRUG CONTRIBUTED': 'DRUGS',
    'MEDICATION PRESENT': 'DRUGS',
    'MEDICATION CONTRIBUTED': 'DRUGS',
    'COMBINED SUBSTANCE PRESENT': 'COMBINED SUBSTANCE',
    'COMBINATION CONTRIBUTED': 'COMBINED SUBSTANCE',
    'SUSPECT OF ALCOHOL USE, SUSPECT OF DRUG USE': 'COMBINED SUBSTANCE',
    'UNKNOWN': 'UNKNOWN/OTHER',
    'UNKNOWN, UNKNOWN': 'UNKNOWN/OTHER',
    'OTHER': 'UNKNOWN/OTHER',
    'UNKNOWN, SUSPECT OF DRUG USE': 'UNKNOWN/OTHER'
    # Add more mappings if needed
})

# Check the unique values after cleaning
print(df['Driver Substance Abuse'].value_counts())


Driver Substance Abuse
NONE/NOT SUSPECTED                                 129749
UNKNOWN/OTHER                                       30271
ALCOHOL                                              5453
DRUGS                                                 501
COMBINED SUBSTANCE                                    149
SUSPECT OF ALCOHOL USE, UNKNOWN                        33
UNKNOWN, NOT SUSPECT OF DRUG USE                       30
NOT SUSPECT OF ALCOHOL USE, SUSPECT OF DRUG USE        12
Name: count, dtype: int64


In [94]:
df['Light'].value_counts()

Light
DAYLIGHT                    114568
DARK LIGHTS ON               35076
DARK NO LIGHTS                4407
DUSK                          3738
DAWN                          3320
DARK - LIGHTED                3011
DARK -- UNKNOWN LIGHTING      1164
DARK - NOT LIGHTED             518
OTHER                          319
DARK - UNKNOWN LIGHTING         64
UNKNOWN                         13
Name: count, dtype: int64

In [95]:
# Replace specific values with broader categories
df['Light'] = df['Light'].replace({
    'DAYLIGHT': 'DAYLIGHT',
    'DARK LIGHTS ON': 'DARK (WITH LIGHTS ON)',
    'DARK - LIGHTED': 'DARK (WITH LIGHTS ON)',
    'DARK NO LIGHTS': 'DARK (NO LIGHTS)',
    'DARK - NOT LIGHTED': 'DARK (NO LIGHTS)',
    'DUSK': 'TRANSITIONAL LIGHTING',
    'DAWN': 'TRANSITIONAL LIGHTING',
    'DARK -- UNKNOWN LIGHTING': 'UNKNOWN/OTHER',
    'DARK - UNKNOWN LIGHTING': 'UNKNOWN/OTHER',
    'OTHER': 'UNKNOWN/OTHER',
    'UNKNOWN': 'UNKNOWN/OTHER'
})

# Check the unique values after cleaning
print(df['Light'].value_counts())


Light
DAYLIGHT                 114568
DARK (WITH LIGHTS ON)     38087
TRANSITIONAL LIGHTING      7058
DARK (NO LIGHTS)           4925
UNKNOWN/OTHER              1560
Name: count, dtype: int64


In [96]:
df['Surface Condition'].value_counts()

Surface Condition
DRY                         119874
WET                          28315
UNKNOWN                      15433
ICE                            977
SNOW                           963
SLUSH                          232
OTHER                          172
ICE/FROST                      113
MUD, DIRT, GRAVEL               45
WATER(STANDING/MOVING)          39
OIL                             27
WATER (STANDING, MOVING)         5
SAND                             3
Name: count, dtype: int64

In [97]:
df['Surface Condition'] = df['Surface Condition'].replace({
    'DRY': 'DRY',
    'WET': 'WET',
    'WATER(STANDING/MOVING)': 'WET',
    'WATER (STANDING, MOVING)': 'WET',
    'ICE': 'ICE/FROST',
    'ICE/FROST': 'ICE/FROST',
    'SNOW': 'SNOW/SLUSH',
    'SLUSH': 'SNOW/SLUSH',
    'UNKNOWN': 'OTHER',
    'OTHER': 'OTHER',
    'MUD, DIRT, GRAVEL': 'OTHER',
    'OIL': 'OTHER',
    'SAND': 'OTHER'
})

# Check the unique values after cleaning
print(df['Surface Condition'].value_counts())


Surface Condition
DRY           119874
WET            28359
OTHER          15680
SNOW/SLUSH      1195
ICE/FROST       1090
Name: count, dtype: int64


In [98]:
df.to_csv('second_cleaned_data.csv', index=False)

In [99]:
df['Weather'].value_counts()

Weather
CLEAR                                114286
RAINING                               18882
CLOUDY                                17035
UNKNOWN                               11337
SNOW                                   1464
RAIN                                   1309
FOGGY                                   622
WINTRY MIX                              355
OTHER                                   331
SLEET                                   202
BLOWING SNOW                            153
SEVERE WINDS                            143
FREEZING RAIN OR FREEZING DRIZZLE        27
FOG, SMOG, SMOKE                         21
BLOWING SAND, SOIL, DIRT                 15
SLEET OR HAIL                             9
SEVERE CROSSWINDS                         7
Name: count, dtype: int64

In [100]:
# Replace specific values with broader categories
df['Weather'] = df['Weather'].replace({
    'CLEAR': 'CLEAR',
    'RAINING': 'RAIN',
    'RAIN': 'RAIN',
    'FREEZING RAIN OR FREEZING DRIZZLE': 'RAIN',
    'CLOUDY': 'CLOUDY',
    'SNOW': 'SNOW',
    'BLOWING SNOW': 'SNOW',
    'FOGGY': 'FOG/SMOG',
    'FOG, SMOG, SMOKE': 'FOG/SMOG',
    'WINTRY MIX': 'WINTRY MIX',
    'SLEET': 'WINTRY MIX',
    'SLEET OR HAIL': 'WINTRY MIX',
    'SEVERE WINDS': 'SEVERE WINDS',
    'SEVERE CROSSWINDS': 'SEVERE WINDS',
    'BLOWING SAND, SOIL, DIRT': 'SEVERE WINDS',
    'UNKNOWN': 'UNKNOWN/OTHER',
    'OTHER': 'UNKNOWN/OTHER'
})

# Check the unique values after cleaning
print(df['Weather'].value_counts())


Weather
CLEAR            114286
RAIN              20218
CLOUDY            17035
UNKNOWN/OTHER     11668
SNOW               1617
FOG/SMOG            643
WINTRY MIX          566
SEVERE WINDS        165
Name: count, dtype: int64


In [101]:
df['Driver Distracted By'].value_counts()

Driver Distracted By
NOT DISTRACTED                                       110038
UNKNOWN                                               24887
LOOKED BUT DID NOT SEE                                20532
INATTENTIVE OR LOST IN THOUGHT                         4062
OTHER DISTRACTION                                      3058
DISTRACTED BY OUTSIDE PERSON OBJECT OR EVENT            940
OTHER ACTION (LOOKING AWAY FROM TASK, ETC.)             439
BY OTHER OCCUPANTS                                      404
OTHER CELLULAR PHONE RELATED                            361
OTHER ELECTRONIC DEVICE (NAVIGATIONAL PALM PILOT)       316
TALKING OR LISTENING TO CELLULAR PHONE                  259
BY MOVING OBJECT IN VEHICLE                             206
EATING OR DRINKING                                      187
ADJUSTING AUDIO AND OR CLIMATE CONTROLS                 131
USING OTHER DEVICE CONTROLS INTEGRAL TO VEHICLE          87
USING DEVICE OBJECT BROUGHT INTO VEHICLE                 62
TEXTING FROM A CELL

In [102]:
# Replace specific values with broader categories
df['Driver Distracted By'] = df['Driver Distracted By'].replace({
    'NOT DISTRACTED': 'NOT DISTRACTED',
    'UNKNOWN': 'UNKNOWN',
    'LOOKED BUT DID NOT SEE': 'INATTENTIVE',
    'INATTENTIVE OR LOST IN THOUGHT': 'INATTENTIVE',
    'DISTRACTED BY OUTSIDE PERSON OBJECT OR EVENT': 'EXTERNAL DISTRACTION',
    'OTHER DISTRACTION': 'EXTERNAL DISTRACTION',
    'TALKING OR LISTENING TO CELLULAR PHONE': 'CELL PHONE/ELECTRONICS',
    'TEXTING FROM A CELLULAR PHONE': 'CELL PHONE/ELECTRONICS',
    'DIALING CELLULAR PHONE': 'CELL PHONE/ELECTRONICS',
    'OTHER CELLULAR PHONE RELATED': 'CELL PHONE/ELECTRONICS',
    'OTHER ELECTRONIC DEVICE (NAVIGATIONAL PALM PILOT)': 'CELL PHONE/ELECTRONICS',
    'BY OTHER OCCUPANTS': 'INSIDE VEHICLE DISTRACTION',
    'BY MOVING OBJECT IN VEHICLE': 'INSIDE VEHICLE DISTRACTION',
    'EATING OR DRINKING': 'INSIDE VEHICLE DISTRACTION',
    'ADJUSTING AUDIO AND OR CLIMATE CONTROLS': 'INSIDE VEHICLE DISTRACTION',
    'USING OTHER DEVICE CONTROLS INTEGRAL TO VEHICLE': 'INSIDE VEHICLE DISTRACTION',
    'USING DEVICE OBJECT BROUGHT INTO VEHICLE': 'INSIDE VEHICLE DISTRACTION',
    'NO DRIVER PRESENT': 'OTHER',
    'TALKING/LISTENING': 'OTHER',
    'SMOKING RELATED': 'OTHER',
    'MANUALLY OPERATING (DIALING, PLAYING GAME, ETC.)': 'OTHER'
})

# Check the unique values after cleaning
print(df['Driver Distracted By'].value_counts())


Driver Distracted By
NOT DISTRACTED                                 110038
UNKNOWN                                         24887
INATTENTIVE                                     24594
EXTERNAL DISTRACTION                             3998
INSIDE VEHICLE DISTRACTION                       1077
CELL PHONE/ELECTRONICS                           1035
OTHER ACTION (LOOKING AWAY FROM TASK, ETC.)       439
OTHER                                             130
Name: count, dtype: int64


In [104]:
df['Vehicle First Impact Location'].value_counts()

Vehicle First Impact Location
TWELVE OCLOCK           62251
SIX OCLOCK              31708
ONE OCLOCK              14392
ELEVEN OCLOCK           12499
TWO OCLOCK               5586
TEN OCLOCK               5538
FOUR OCLOCK              3822
TWELVE O CLOCK           3820
SEVEN OCLOCK             3622
FIVE OCLOCK              3579
EIGHT OCLOCK             3403
THREE OCLOCK             3143
NINE OCLOCK              3080
SIX O CLOCK              2059
ONE O CLOCK              1396
ELEVEN O CLOCK           1377
TEN O CLOCK               531
UNDERSIDE                 520
TWO O CLOCK               500
SEVEN O CLOCK             447
NINE O CLOCK              437
NON-COLLISION             424
FIVE O CLOCK              413
THREE O CLOCK             411
FOUR O CLOCK              353
EIGHT O CLOCK             332
ROOF TOP                  308
VEHICLE NOT AT SCENE      136
TOP                        92
CARGO LOSS                 19
Name: count, dtype: int64

In [105]:
# Replace specific values with broader categories
df['Vehicle First Impact Location'] = df['Vehicle First Impact Location'].replace({
    # FRONT
    'TWELVE OCLOCK': 'FRONT',
    'TWELVE O CLOCK': 'FRONT',

    # REAR
    'SIX OCLOCK': 'REAR',
    'SIX O CLOCK': 'REAR',

    # SIDES (LEFT)
    'ELEVEN OCLOCK': 'SIDES (LEFT)',
    'ELEVEN O CLOCK': 'SIDES (LEFT)',
    'TEN OCLOCK': 'SIDES (LEFT)',
    'TEN O CLOCK': 'SIDES (LEFT)',
    'NINE OCLOCK': 'SIDES (LEFT)',
    'NINE O CLOCK': 'SIDES (LEFT)',

    # SIDES (RIGHT)
    'ONE OCLOCK': 'SIDES (RIGHT)',
    'ONE O CLOCK': 'SIDES (RIGHT)',
    'TWO OCLOCK': 'SIDES (RIGHT)',
    'TWO O CLOCK': 'SIDES (RIGHT)',
    'THREE OCLOCK': 'SIDES (RIGHT)',
    'THREE O CLOCK': 'SIDES (RIGHT)',

    # CORNER IMPACTS
    'FOUR OCLOCK': 'CORNER IMPACTS',
    'FOUR O CLOCK': 'CORNER IMPACTS',
    'FIVE OCLOCK': 'CORNER IMPACTS',
    'FIVE O CLOCK': 'CORNER IMPACTS',
    'SEVEN OCLOCK': 'CORNER IMPACTS',
    'SEVEN O CLOCK': 'CORNER IMPACTS',
    'EIGHT OCLOCK': 'CORNER IMPACTS',
    'EIGHT O CLOCK': 'CORNER IMPACTS',

    # ROOF/TOP
    'ROOF TOP': 'ROOF/TOP',
    'TOP': 'ROOF/TOP',

    # UNDERSIDE
    'UNDERSIDE': 'UNDERSIDE',

    # NON-COLLISION
    'NON-COLLISION': 'NON-COLLISION',
    'CARGO LOSS': 'NON-COLLISION',

    # VEHICLE NOT AT SCENE
    'VEHICLE NOT AT SCENE': 'VEHICLE NOT AT SCENE',

    # OTHER
    'OTHER': 'OTHER'  # Rare or ambiguous cases
})

# Check the unique values after cleaning
print(df['Vehicle First Impact Location'].value_counts())


Vehicle First Impact Location
FRONT                   66071
REAR                    33767
SIDES (RIGHT)           25428
SIDES (LEFT)            23462
CORNER IMPACTS          15971
UNDERSIDE                 520
NON-COLLISION             443
ROOF/TOP                  400
VEHICLE NOT AT SCENE      136
Name: count, dtype: int64


In [106]:
df['Vehicle Body Type'].value_counts()

Vehicle Body Type
PASSENGER CAR                                                116932
(SPORT) UTILITY VEHICLE                                       14785
PICKUP TRUCK                                                   5963
VAN                                                            4455
TRANSIT BUS                                                    3476
SCHOOL BUS                                                     2759
SPORT UTILITY VEHICLE                                          2134
POLICE VEHICLE/NON EMERGENCY                                   1936
OTHER LIGHT TRUCKS (10,000LBS (4,536KG) OR LESS)               1677
CARGO VAN/LIGHT TRUCK 2 AXLES (OVER 10,000LBS (4,536 KG))      1600
OTHER                                                          1470
POLICE VEHICLE/EMERGENCY                                       1374
MEDIUM/HEAVY TRUCKS 3 AXLES (OVER 10,000LBS (4,536KG))         1294
MOTORCYCLE                                                      791
STATION WAGON                 

In [107]:
# Replace specific values with broader categories
df['Vehicle Body Type'] = df['Vehicle Body Type'].replace({
    # PASSENGER CAR
    'PASSENGER CAR': 'PASSENGER CAR',
    'STATION WAGON': 'PASSENGER CAR',

    # SPORT UTILITY VEHICLE (SUV)
    '(SPORT) UTILITY VEHICLE': 'SPORT UTILITY VEHICLE (SUV)',
    'SPORT UTILITY VEHICLE': 'SPORT UTILITY VEHICLE (SUV)',

    # PICKUP TRUCK
    'PICKUP TRUCK': 'PICKUP TRUCK',
    'PICKUP': 'PICKUP TRUCK',

    # VAN
    'VAN': 'VAN',
    'VAN - PASSENGER (<9 SEATS)': 'VAN',
    'VAN - PASSENGER (9 OR 12 SEATS)': 'VAN',
    'VAN - PASSENGER (15 SEATS)': 'VAN',
    'VAN - CARGO': 'VAN',

    # TRUCK
    'MEDIUM/HEAVY TRUCKS 3 AXLES (OVER 10,000LBS (4,536KG))': 'TRUCK',
    'CARGO VAN/LIGHT TRUCK 2 AXLES (OVER 10,000LBS (4,536 KG))': 'TRUCK',
    'SINGLE-UNIT TRUCK': 'TRUCK',
    'TRUCK TRACTOR': 'TRUCK',

    # EMERGENCY VEHICLE
    'POLICE VEHICLE/NON EMERGENCY': 'EMERGENCY VEHICLE',
    'POLICE VEHICLE/EMERGENCY': 'EMERGENCY VEHICLE',
    'AMBULANCE/EMERGENCY': 'EMERGENCY VEHICLE',
    'FIRE VEHICLE/EMERGENCY': 'EMERGENCY VEHICLE',
    'FIRE VEHICLE/NON EMERGENCY': 'EMERGENCY VEHICLE',
    'AMBULANCE/NON EMERGENCY': 'EMERGENCY VEHICLE',

    # BUS
    'TRANSIT BUS': 'BUS',
    'SCHOOL BUS': 'BUS',
    'OTHER BUS': 'BUS',
    'BUS - TRANSIT': 'BUS',
    'BUS - SCHOOL': 'BUS',
    'BUS - MINI': 'BUS',
    'BUS - OTHER TYPE': 'BUS',
    'BUS - CROSS COUNTRY': 'BUS',

    # MOTORCYCLE
    'MOTORCYCLE': 'MOTORCYCLE',
    'MOTORCYCLE - 2 WHEELED': 'MOTORCYCLE',
    'MOTORCYCLE - 3 WHEELED': 'MOTORCYCLE',
    'MOPED': 'MOTORCYCLE',
    'MOPED OR MOTORIZED BICYCLE': 'MOTORCYCLE',

    # RECREATIONAL VEHICLE (RV)
    'RECREATIONAL VEHICLE': 'RECREATIONAL VEHICLE',
    'RECREATIONAL OFF-HIGHWAY VEHICLES (ROV)': 'RECREATIONAL VEHICLE',

    # ALL-TERRAIN VEHICLE (ATV)
    'ALL TERRAIN VEHICLE (ATV)': 'ALL-TERRAIN VEHICLE (ATV)',
    'ALL-TERRAIN VEHICLE/ALL-TERRAIN CYCLE (ATV/ATC)': 'ALL-TERRAIN VEHICLE (ATV)',
    'SNOWMOBILE': 'ALL-TERRAIN VEHICLE (ATV)',

    # OTHER
    'OTHER': 'OTHER',
    'UNKNOWN': 'OTHER',
    'FARM VEHICLE': 'OTHER',
    'LIMOUSINE': 'OTHER',
    'CROSS COUNTRY BUS': 'OTHER',
    'CONSTRUCTION EQUIPMENT (BACKHOE, BULLDOZER, ETC.)': 'OTHER',
    'GOLF CART': 'OTHER'
})

# Check the unique values after cleaning
print(df['Vehicle Body Type'].value_counts())


Vehicle Body Type
PASSENGER CAR                                        117677
SPORT UTILITY VEHICLE (SUV)                           16919
BUS                                                    7323
PICKUP TRUCK                                           6613
EMERGENCY VEHICLE                                      4573
VAN                                                    4539
TRUCK                                                  3461
OTHER LIGHT TRUCKS (10,000LBS (4,536KG) OR LESS)       1677
OTHER                                                  1545
MOTORCYCLE                                              937
VAN - PASSENGER (&LT;9 SEATS)                           300
ALL-TERRAIN VEHICLE (ATV)                               232
RECREATIONAL VEHICLE                                    185
OTHER TRUCKS                                            143
AUTOCYCLE                                                39
LOW SPEED VEHICLE                                        33
FARM EQUIPMENT (TRACTO

In [108]:
# Update the 'Vehicle Body Type' column with final groupings
df['Vehicle Body Type'] = df['Vehicle Body Type'].replace({
    # VAN GROUPING
    'VAN - PASSENGER (&LT;9 SEATS)': 'VAN',

    # TRUCK GROUPING
    'OTHER LIGHT TRUCKS (10,000LBS (4,536KG) OR LESS)': 'TRUCK',

    # OTHER GROUPING
    'AUTOCYCLE': 'OTHER',
    'LOW SPEED VEHICLE': 'OTHER',
    'FARM EQUIPMENT (TRACTOR, COMBINE HARVESTER, ETC.)': 'OTHER',
    'OTHER TRUCKS': 'OTHER'
})

# Check the updated counts
print(df['Vehicle Body Type'].value_counts())



Vehicle Body Type
PASSENGER CAR                  117677
SPORT UTILITY VEHICLE (SUV)     16919
BUS                              7323
PICKUP TRUCK                     6613
TRUCK                            5138
VAN                              4839
EMERGENCY VEHICLE                4573
OTHER                            1762
MOTORCYCLE                        937
ALL-TERRAIN VEHICLE (ATV)         232
RECREATIONAL VEHICLE              185
Name: count, dtype: int64


In [109]:
df['Vehicle Movement'].value_counts()

Vehicle Movement
MOVING CONSTANT SPEED      67807
SLOWING OR STOPPING        25039
STOPPED IN TRAFFIC LANE    17992
MAKING LEFT TURN           15799
ACCELERATING                9570
MAKING RIGHT TURN           4861
BACKING                     4727
CHANGING LANES              4391
STARTING FROM LANE          4281
STARTING FROM PARKED        1486
TURNING LEFT                1369
PARKING                     1342
STOPPED IN TRAFFIC          1212
MAKING U TURN               1130
ENTERING TRAFFIC LANE       1034
PASSING                      780
OTHER                        776
NEGOTIATING A CURVE          679
SKIDDING                     658
TURNING RIGHT                465
LEAVING TRAFFIC LANE         234
RIGHT TURN ON RED            232
OVERTAKING/PASSING           140
MAKING U-TURN                121
PARKED                        41
DRIVERLESS MOVING VEH.        32
Name: count, dtype: int64

In [110]:
df.to_csv('Third_cleaned_data.csv', index=False)

In [112]:
df['Collision Type'].value_counts()

Collision Type
SAME DIR REAR END                51532
STRAIGHT MOVEMENT ANGLE          29020
OTHER                            14404
SINGLE VEHICLE                   14312
SAME DIRECTION SIDESWIPE         13328
HEAD ON LEFT TURN                12561
Front to Rear                     4045
HEAD ON                           3399
SAME DIRECTION LEFT TURN          3379
SAME DIRECTION RIGHT TURN         3338
OPPOSITE DIRECTION SIDESWIPE      2393
Angle                             2247
ANGLE MEETS LEFT TURN             1947
Sideswipe, Same Direction         1902
Single Vehicle                    1556
ANGLE MEETS RIGHT TURN            1147
Other                              787
Front to Front                     730
SAME DIR REND LEFT TURN            672
ANGLE MEETS LEFT HEAD ON           661
SAME DIR REND RIGHT TURN           647
Rear To Side                       619
SAME DIR BOTH LEFT TURN            612
Sideswipe, Opposite Direction      465
OPPOSITE DIR BOTH LEFT TURN        299
Rear To Re

In [113]:
# Refine Collision Type column
df['Collision Type'] = df['Collision Type'].replace({
    # Combine synonyms
    'SAME DIR REAR END': 'REAR END',
    'Front to Rear': 'REAR END',
    'SAME DIRECTION SIDESWIPE': 'SIDESWIPE SAME DIRECTION',
    'Sideswipe, Same Direction': 'SIDESWIPE SAME DIRECTION',
    'OPPOSITE DIRECTION SIDESWIPE': 'SIDESWIPE OPPOSITE DIRECTION',
    'Sideswipe, Opposite Direction': 'SIDESWIPE OPPOSITE DIRECTION',
    'SINGLE VEHICLE': 'SINGLE VEHICLE',
    'Single Vehicle': 'SINGLE VEHICLE',
    'STRAIGHT MOVEMENT ANGLE': 'STRAIGHT ANGLE',
    'Angle': 'STRAIGHT ANGLE',
    'HEAD ON LEFT TURN': 'HEAD ON',
    'HEAD ON': 'HEAD ON',

    # Simplify phrasing
    'ANGLE MEETS LEFT TURN': 'LEFT TURN ANGLE',
    'ANGLE MEETS RIGHT TURN': 'RIGHT TURN ANGLE',
    'SAME DIRECTION LEFT TURN': 'LEFT TURN',
    'SAME DIRECTION RIGHT TURN': 'RIGHT TURN',

    # Group rare or ambiguous categories into OTHER
    'Unknown': 'OTHER',
    'Rear To Side': 'OTHER',
    'Rear To Rear': 'OTHER',
    'ANGLE MEETS LEFT HEAD ON': 'OTHER',
    'SAME DIR REND LEFT TURN': 'OTHER',
    'SAME DIR REND RIGHT TURN': 'OTHER',
    'SAME DIR BOTH LEFT TURN': 'OTHER',
    'OPPOSITE DIR BOTH LEFT TURN': 'OTHER'
})

# Check updated counts
print(df['Collision Type'].value_counts())



Collision Type
REAR END                        55577
STRAIGHT ANGLE                  31267
OTHER                           18110
HEAD ON                         15960
SINGLE VEHICLE                  15868
SIDESWIPE SAME DIRECTION        15230
LEFT TURN                        3379
RIGHT TURN                       3338
SIDESWIPE OPPOSITE DIRECTION     2858
LEFT TURN ANGLE                  1947
RIGHT TURN ANGLE                 1147
Other                             787
Front to Front                    730
Name: count, dtype: int64


In [114]:

df['Crash Date/Time'].value_counts()

Crash Date/Time
12/10/2018 06:10:00 PM    11
03/03/2017 06:00:00 AM    10
06/09/2020 06:53:00 PM    10
03/28/2019 09:30:00 AM    10
06/15/2021 02:02:00 PM     9
                          ..
07/30/2018 03:23:00 AM     1
09/21/2018 09:00:00 PM     1
07/14/2016 04:13:00 PM     1
08/18/2016 10:41:00 PM     1
01/17/2025 07:05:00 AM     1
Name: count, Length: 94768, dtype: int64

In [115]:
# 1. Convert 'Crash Date/Time' to datetime format
df['Crash Date/Time'] = pd.to_datetime(df['Crash Date/Time'], errors='coerce')

# 2. Check for any conversion errors
missing_dates = df['Crash Date/Time'].isnull().sum()
print(f"Number of rows with invalid 'Crash Date/Time': {missing_dates}")


/var/folders/n_/fvk81crd6fddbjh6csc8kmt80000gn/T/ipykernel_46631/722324615.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Crash Date/Time'] = pd.to_datetime(df['Crash Date/Time'], errors='coerce')


Number of rows with invalid 'Crash Date/Time': 0


In [116]:
# Optionally, handle rows with invalid datetime
# For example, drop them:
df = df.dropna(subset=['Crash Date/Time'])

# Or fill with a default value if appropriate:
# df['Crash Date/Time'] = df['Crash Date/Time'].fillna(pd.Timestamp('1900-01-01'))

# 3. Create 'Crash Date' and 'Crash Time' columns
df['Crash Date'] = df['Crash Date/Time'].dt.date
df['Crash Time'] = df['Crash Date/Time'].dt.time


In [117]:
df.isnull().sum()
df.shape

(166198, 33)

In [118]:
# Drop the original 'Crash Date/Time' column
df = df.drop(columns=['Crash Date/Time'])
df.isnull().sum()

Report Number                    0
Local Case Number                0
Agency Name                      0
ACRS Report Type                 0
Route Type                       0
Collision Type                   0
Weather                          0
Surface Condition                0
Light                            0
Traffic Control                  0
Driver Substance Abuse           0
Person ID                        0
Driver At Fault                  0
Injury Severity                  0
Driver Distracted By             0
Drivers License State            0
Vehicle ID                       0
Vehicle Damage Extent            0
Vehicle First Impact Location    0
Vehicle Body Type                0
Vehicle Movement                 0
Vehicle Going Dir                0
Speed Limit                      0
Driverless Vehicle               0
Parked Vehicle                   0
Vehicle Year                     0
Vehicle Make                     0
Vehicle Model                    0
Latitude            

In [119]:
df.to_csv('Fourth_cleaned_data.csv', index=False)